In [1]:
from google.colab import files
up = files.upload()   # pick your activations npz

Saving activations_layer20.npz to activations_layer20.npz


In [2]:
import numpy as np

NPZ = '/content/activations_layer20.npz'
data = np.load(NPZ, allow_pickle=True)
print('keys:', list(data.keys()))
for k in data.keys():
    print(f'  {k}: shape={data[k].shape}, dtype={data[k].dtype}')
print('unique tiers :', np.unique(data['tiers']))
print('unique labels:', np.unique(data['labels']))
print('refused in t3:', ((data['tiers']=='tier_3') & (data['labels']=='refused')).sum())

keys: ['scenario_ids', 'activations', 'tiers', 'labels']
  scenario_ids: shape=(496,), dtype=int64
  activations: shape=(496, 3584), dtype=float32
  tiers: shape=(496,), dtype=<U7
  labels: shape=(496,), dtype=<U11
unique tiers : ['tier_1' 'tier_2a' 'tier_2b' 'tier_3' 'tier_4']
unique labels: ['appropriate' 'leaked' 'refused']
refused in t3: 36


In [3]:
import numpy as np, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

NPZ      = '/content/activations_layer20.npz'
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
TOP_K    = 40

data = np.load(NPZ, allow_pickle=True)
t3 = data['tiers'] == 'tier_3'
acts, labels = data['activations'][t3], data['labels'][t3]

def safe_dir(mask):
    a, b = acts[mask], acts[~mask]
    if a.shape[0]==0 or b.shape[0]==0: return None
    return a.mean(0) - b.mean(0)

dirs = {}
for name, mask in [('diff_leaked_vs_not', labels=='leaked'),
                   ('refusal_dir',        labels=='refused')]:
    d = safe_dir(mask)
    if d is not None: dirs[name] = d
    else: print(f'WARNING: skipping {name} (empty group)')

X = acts - acts.mean(0)
_, _, Vt = np.linalg.svd(X, full_matrices=False)
for i in range(3): dirs[f'pc{i+1}'] = Vt[i]

print(f'Loading {MODEL_ID} (bf16)...')
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto').eval()
norm, head = model.model.norm, model.lm_head

for name, v in dirs.items():
    vt = torch.tensor(v/np.linalg.norm(v), dtype=torch.bfloat16, device=model.device)
    with torch.no_grad():
        logits = head(norm(vt.unsqueeze(0)))[0].float()
    top = torch.topk(logits, TOP_K).indices.tolist()
    bot = torch.topk(-logits, TOP_K).indices.tolist()
    fmt = lambda ids: ', '.join(repr(tok.decode([t])) for t in ids)
    print(f'\n{"="*70}\n[{name}]')
    print(f'  +dir: {fmt(top)}')
    print(f'  -dir: {fmt(bot)}')

Loading Qwen/Qwen2.5-7B-Instruct (bf16)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]


[diff_leaked_vs_not]
  +dir: 'ificate', ' Smarty', 'ifestyles', ' CONTRIBUTORS', '不公平', '_ARGUMENT', '自豪', '(nonatomic', '.argument', ' comeback', ' PROGMEM', 'тради', 'groundColor', '不在乎', '惬意', 'jay', '我爸', '反驳', ' QIcon', ' ///<', '反正', '.sav', '.orange', '追捧', 'lords', '豨', '舒服', '纠正', 'благо', 'Wal', '榜样', '生活水平', '贡献', '뤘', '.VAL', '躔', ' 스스', ".'/'.$", ' offsetX', '上年同期'
  -dir: '严格的', '严禁', '处理', 'gfx', '蒟', '��', 'ɨ', 'amina', '+Sans', '绝不', ' ?><?', 'cern', 'Cpp', 'irates', '.titleLabel', ' pBuffer', '严格', 'чки', '实', '熊', ' approaching', '-NLS', ' ?><', '.contentMode', 'lut', ' خط', 'ITES', 'Mathf', 'örü', '芬', '��', '问', ' clearColor', ');}\n\n', ' REGARD', '-repeat', 'ISOString', '-buttons', '治理', '純'

[refusal_dir]
  +dir: 'أوض', '严格', 'oplay', 'anmar', ' Annunci', 'noinspection', ' piger', '\ufeff/*\n', '��', 'nü', '睃', ' frosting', '膘', '.Accessible', 'ISOString', 'ditor', ')){\n\n', ');}\n\n', '拒', 'edm', 'gages', 'cold', 'izzato', ' Đề', ' blev', '专业从事', 'berra', 'or